# 13 — Xuất bản release sang TensorFlow Lite

Đọc `Models/<ts>/release/` do **12** tạo (`sdc_os.onnx` + `manifest.json`), dựng lại đúng mô hình đó bằng graph
TensorFlow, convert sang `.tflite`, kiểm **từng nhãn** khớp `sdc_os.onnx`, rồi ghi `sdc_os.tflite` vào cùng thư mục
và thêm nó vào manifest.

| | `sdc_os.onnx` (12) | `sdc_os.tflite` (notebook này) |
|---|---|---|
| input | `input` `string [N,12]` + `thresholds` (tuỳ chọn) | `input` `string [N,12]` |
| output | `output` `string [N,3]` | giống hệt — `make`, `type`, `model` |
| ngưỡng iden | ghi đè được lúc chạy | **nung cố định** (TFLite không có input mặc định) |
| contract | `metadata_props["sdc.contract"]` | metadata `sdc.contract` trong flatbuffer |
| runtime | onnxruntime | LiteRT **+ Flex delegate** |

**Vì sao cần Flex.** Đường 12 chuỗi cần tách token (`dhcp_prl`, TF-IDF), regex RE2 (luật hostname) và
ghép chuỗi (khoá vân tay L1) — TFLite builtin không có op nào trong số đó. Bật `SELECT_TF_OPS` thì các op
này chạy bằng kernel TensorFlow nhúng trong delegate; mọi thứ còn lại (rừng, hierarchy, ngưỡng, cascade, tra
nhãn) vẫn là op builtin. Runtime phía thiết bị phải link thêm Flex delegate — xem mục 6.

**Không đào lại bảng nào.** Rừng và encoder lấy từ `train/iden.joblib` và `train/os.joblib` của cùng run; mọi bảng của
head iden (vân tay L1, OUI, mDNS `model=`, luật hostname theo thứ tự ưu tiên, mask hierarchy, ngưỡng, nhãn) trích
thẳng từ graph `sdc_iden.onnx`; luật OS và cascade lấy từ mục **Thư viện** đầu notebook.

Cần `tensorflow` (Python 3.14: `pip install --pre tensorflow`), `onnx`, `onnxruntime`; `VERIFY = True` cần thêm `pyarrow`.

In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("Configured for 88-core CPU")

Configured for 88-core CPU


## Thư viện

Hàm và hằng dùng chung, chép **nguyên văn** (không còn notebook `00_common`). Notebook khác có bản sao của cùng hàm — sửa ở đây thì sửa cả các bản kia cho khớp: bảng *Hàm có bản sao ở đâu* trong `README.md`. Phần riêng của notebook này bắt đầu sau mục này.

In [2]:
# --- 1. Nền: đường dẫn, lược đồ, mã hoá ---
# Xác định gốc dự án từ thư mục làm việc của kernel Jupyter.
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ bên trong dự án SDC")
print("Gốc dự án:", ROOT)

Gốc dự án: D:\01.AI_Security\02.SDC


In [3]:
# --- Đường dẫn dự án ---
from pathlib import Path


DATA = ROOT / "Data"
MODELS = ROOT / "Models"
SESSIONS_PATH = DATA / "sessions_verified.parquet"

In [4]:
# --- Lược đồ dữ liệu — head, nhóm feature, chia tập ---
import pandas as pd


MISSING = "<missing>"

META_COLS = ["canonical_device", "scenario", "capture_file", "date", "mac", "n_sources"]

# Ba head, đúng thứ tự output của sản phẩm: hãng / loại / model sản phẩm.
#
# Head thứ ba từng tên là `family` (contract ≤ 2.1). Đổi thành `model` cho khớp ô MODEL
# trong spec, nhưng chỉ là ĐỔI TÊN — nội dung nhãn không đổi, và độ mịn của nó vẫn là
# "model sản phẩm ở mức phân biệt được bằng telemetry": `Amazon Echo Dot` là một model,
# còn `Tuya Plug` vẫn gộp 13 thiết bị ODM không tách được bằng DHCP/DNS/TLS. Đổi tên
# không làm chúng tách ra được; xem `Data/device_labels_verified.csv` cột review_note.
MODEL_HEAD = "model"
LEGACY_MODEL_HEAD = "family"
HEAD_ALIASES = {LEGACY_MODEL_HEAD: MODEL_HEAD}
LABEL_COLS = ["make", "type", MODEL_HEAD]
SOURCES = ["dhcp", "dns", "mdns", "tls"]

# Cột dạng chuỗi token -> TF-IDF. Phần còn lại dạng chuỗi -> categorical nguyên giá trị.
TEXT_COLS = ["dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens"]

# Feature biến mất khi DNS được mã hoá (DoH/DoT) và SNI bị ẩn (ECH). Đây là lý do thật
# để đo bản không có chúng — không phải vì sợ leak: lúc chạy thật router vẫn đọc được
# DNS query và SNI, nên chúng là bằng chứng hợp lệ.
ENCRYPTED_RISK = ["dns_tokens", "tls_sni_tokens"]

# Field driver KHÔNG cung cấp — không feature nào được suy ra từ chúng
FORBIDDEN = ["src_ip", "dst_ip", "dst_port", "ts", "iat", "pktsize", "bytes", "rate"]


# Cot co-mat cua tung nguon. Suy tu `SOURCES` de danh sach nguon chi co MOT ban.
SOURCE_FLAGS = tuple(f"has_{source}" for source in SOURCES)


def feature_sets(df):
    """Các nhóm feature, suy từ chính các cột của dataframe."""
    feats = [c for c in df.columns if c not in META_COLS + LABEL_COLS]
    text = [c for c in TEXT_COLS if c in feats]
    cat = [c for c in feats if c not in text and not pd.api.types.is_numeric_dtype(df[c])]
    num = [c for c in feats if c not in text and c not in cat]
    return {
        "all": feats,
        "text": text,
        "cat": cat,
        "num": num,
        "source": {s: [c for c in feats if c.startswith(s)] for s in SOURCES},
        "no_content": [c for c in feats if c not in ENCRYPTED_RISK],
    }


def assert_serveable(cols):
    """Fail nếu có feature vượt quá telemetry driver cung cấp."""
    bad = [c for c in cols if any(f in c for f in FORBIDDEN)]
    assert not bad, f"Feature không phục vụ được lúc chạy thật: {bad}"
    orphan = [c for c in cols if not c.startswith(("dhcp_", "dns_", "mdns_", "tls_", "has_"))]
    assert not orphan, f"Feature không truy được về nguồn nào: {orphan}"
    return True


def normalize_heads(df):
    """Đổi tên cột nhãn cũ sang tên head hiện hành (`family` -> `model`).

    Làm ở đây, một chỗ, thay vì ghi lại parquet: notebook 03_build_dataset và các file dữ liệu đã sinh
    ra từ trước vẫn dùng được, mà phần còn lại của code chỉ biết đúng một tên. Cột mới
    đã có sẵn thì không đụng vào — dữ liệu mới luôn thắng.
    """
    rename = {old: new for old, new in HEAD_ALIASES.items()
              if old in df.columns and new not in df.columns}
    # Có cả hai cột: cột mới là nhãn đang dùng, cột cũ là tàn dư của file dựng trước lần
    # đổi tên. Bỏ hẳn cột cũ — để lại thì nó không phải nhãn mà cũng không phải feature,
    # và `assert_serveable` sẽ bắt nó như một feature không truy được về nguồn nào.
    stale = [old for old, new in HEAD_ALIASES.items()
             if old in df.columns and new in df.columns]
    if rename or stale:
        df = df.rename(columns=rename).drop(columns=stale)
    return df


def load_sessions(path=SESSIONS_PATH):
    """Nạp feature table mức session kèm nhóm feature đã kiểm tra."""
    assert path.exists(), f"Không tìm thấy {path} — chạy 03_build_dataset.ipynb trước"
    df = normalize_heads(pd.read_parquet(path))
    fs = feature_sets(df)
    assert_serveable(fs["all"])
    return df, fs

In [5]:
# --- Bộ mã hoá feature (TF-IDF + ordinal) ---
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OrdinalEncoder


def prepare_frame(df, encoder):
    """Ép kiểu đúng như lúc fit: numeric float32, categorical/text chuỗi.

    ColumnTransformer khớp cột theo TÊN nên thứ tự cột đầu vào không quan trọng, nhưng
    kiểu thì có: TF-IDF nhận object/str, để lọt NaN vào là vỡ.
    """
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    out = df[num + cat + text].copy()
    for c in num:
        out[c] = out[c].astype(np.float32)
    for c in cat + text:
        out[c] = out[c].astype(str)
    return out


def encoder_vectorizers(encoder):
    """TF-IDF đã fit, theo cột — 05 cần để ghi tham số vào contract."""
    named = encoder["ct"].named_transformers_
    return {c: named[f"t_{c}"] for c in encoder["text_cols"]}


def apply_encoder(df, encoder):
    """Trả (X, feature_names, cat_index). cat_index dùng cho HistGradientBoosting."""
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    X = encoder["ct"].transform(prepare_frame(df, encoder)).astype(np.float32)
    names = list(num) + list(cat)
    vecs = encoder_vectorizers(encoder)
    for c in text:
        names += [f"{c}::{t}" for t in vecs[c].get_feature_names_out()]
    cat_index = list(range(len(num), len(num) + len(cat)))
    return X, names, cat_index

In [6]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Hằng của contract `sdc_iden.onnx` (09). 12/13 ghép head OS vào chính graph này nên dùng chung.
import hashlib

IDEN_FORMAT = "sdc-iden-onnx-v2"
UNKNOWN_LABEL = "__unknown__"
KEY_SEP = " || "   # nối các cột thành khoá vân tay L1
NOT_PRESENT = ("", MISSING, "nan", "None")

HEADS = ("make", "type", "model")

IN_FEATURES = "input"

# Output: đúng MỘT tensor `string [N,3]`, cột theo thứ tự HEADS.
#     output[:, 0] = make   "Generic Laptop"
#     output[:, 1] = type   "Laptop"
#     output[:, 2] = model  "Windows Laptop HP"
# Không trả confidence / topk / retrieval. Xác suất, thứ hạng và tầng nào trả lời đều
# chỉ tồn tại BÊN TRONG graph.
OUT_LABELS = "output"

# Hệ quả của việc bỏ output `abstain`: bảng ngưỡng sẽ thành đồ chết nếu không có chỗ
# nào biểu đạt "không đủ tin". Nên abstain được ghi thẳng vào nhãn: dưới ngưỡng thì head
# trả `__unknown__`. Núm tune ngưỡng vẫn sống, đổi lại mất khả năng phân biệt "rừng tự
# nói không biết" với "có đoán nhưng không đủ tin" — hai thứ mà contract closedset cũ cố
# ý tách. Muốn tách lại thì phải thêm output thứ tư.
ABSTAIN_LABEL = UNKNOWN_LABEL

# Phá hoà xếp hạng. Xác suất của rừng có RẤT nhiều giá trị bằng nhau (lớp không lá nào
# bầu đều được 0.0), và ArgMax của onnxruntime với `argmax` của numpy không bảo đảm phá
# hoà giống nhau. Trừ đi một lượng tăng dần theo chỉ số lớp trước khi xếp hạng biến "ai
# thắng khi hoà" thành luật của MÌNH (chỉ số nhỏ thắng), không còn phụ thuộc bản
# onnxruntime trên router. Nhãn trả ra không đổi vì khoản trừ chỉ dùng để xếp hạng.
TIEBREAK = 1e-6

# Input: 12 cột chuỗi, đúng thứ tự này. Không cột số nào cả — 31 cột số mà encoder cần
# (`has_*`, `dhcp_opt_*`, `dhcp_prl_len`) do graph tự tính từ các cột này, xem
# `build_derived`. Caller không còn chỗ nào để tính cờ lệch với lúc train.
ENCODER_STRINGS = ("dhcp_prl", "tls_fp", "tls_version", "tls_alpn", "tls_ciphers",
                   "dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens")
L0_COLUMNS = ("mac_oui", "dhcp_hostname", "mdns_model")
INPUT_COLUMNS = ENCODER_STRINGS + L0_COLUMNS
# Nguồn có mặt <=> một trong các cột chuỗi của nó có giá trị. `aggregate()` đặt cờ theo
# việc CÓ BẢN GHI, nên hai cách chỉ lệch khi có bản ghi mà không trích được gì — đo ở
# cell "Cờ tự tính so với cờ gốc".
SOURCE_EVIDENCE = {
    # KHÔNG gồm `dhcp_hostname`: agent router lấp cột đó bằng tên từ mDNS/NetBIOS khi cửa
    # sổ không có gói DHCP (`Code/router/hostname_sources.py`). Tính nó vào đây là bật
    # `has_dhcp` cho cửa sổ không có DHCP — đúng điều file đó cấm.
    "has_dhcp": ("dhcp_prl", "dhcp_vci"),
    "has_dns": ("dns_tokens",),
    "has_mdns": ("mdns_tokens",),
    "has_tls": ("tls_fp",),
}
# PRL phải ở dạng chuẩn của extractor: số thập phân cách nhau bởi dấu phẩy, không khoảng
# trắng, không số 0 đầu (`1,3,6,15`). `aggregate()` lấy mọi dãy chữ số bằng regex, còn
# graph không có findall nên tách theo dấu phẩy và so chuỗi — hai cách chỉ trùng nhau
# trên dạng chuẩn.
PRL_SEP = ","

FP_MODES = (
    ("fp_full", ("dhcp_prl", "dhcp_vci", "tls_fp"), ("has_dhcp", "has_tls")),
    ("fp_dhcp", ("dhcp_prl", "dhcp_vci"), ("has_dhcp",)),
    ("fp_tls", ("tls_fp",), ("has_tls",)),
)

In [7]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
def pick_run(run=None, needs="train/iden.joblib"):
    """Thư mục run `Models/<ts>/`: `run` = tên thư mục, None = run mới nhất có file `needs`.

    Một run gom đầu ra của cả pipeline: `train/` (05), `iden/` (09), `release/` (12, 13). Các bước 09, 12, 13
    chọn run bằng biến `RUN = None` đặt ở đầu mỗi notebook, nên mở notebook nào chạy riêng cũng biết nó
    đang đọc run nào.
    """
    if run:
        path = MODELS / run
        assert (path / needs).is_file(), f"{path} không có {needs}"
        return path
    runs = sorted(p for p in MODELS.iterdir() if (p / needs).is_file())
    assert runs, f"Không thấy run nào có {needs} trong {MODELS}"
    return runs[-1]


# Bản deploy: 12 ghi `Models/<ts>/release/` của run (chạy lại 12 thì thay cả thư mục); 13 thêm bản TFLite
# vào đúng thư mục đó. `manifest.json` ghi run nguồn và sha256 của từng file.
RELEASE_ONNX = "sdc_os.onnx"
RELEASE_TFLITE = "sdc_os.tflite"
RELEASE_MANIFEST = "manifest.json"

In [8]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Chuẩn bị input 12 cột — bản Python của phần graph tự tính (`build_derived` ở 09).
import re

import numpy as np
import pandas as pd


_MAC_RE = re.compile(r"^[0-9a-f]{2}(:[0-9a-f]{2}){5}$")


def canonical_mac(value):
    if value is None:
        return MISSING
    text = str(value).strip().lower().replace("-", ":")
    return text if _MAC_RE.match(text) else MISSING


def mac_is_random(value):
    """Bit U/L (0x02) của octet đầu: 1 = locally administered = MAC ngẫu nhiên hoá."""
    mac = canonical_mac(value)
    return 0 if mac == MISSING else int(int(mac[:2], 16) & 0x02 != 0)


def mac_oui(value):
    """3 octet đầu. MAC ngẫu nhiên hoá trả <missing> — OUI của nó vô nghĩa."""
    mac = canonical_mac(value)
    if mac == MISSING or mac_is_random(mac):
        return MISSING
    return mac[:8]


def _clean_text(value):
    if value is None:
        return MISSING
    text = str(value).strip().strip(".")
    return text if text and text not in NOT_PRESENT else MISSING


def add_l0_columns(frame):
    """Bổ sung 4 cột L0. Cột nào extractor chưa sinh thì <missing> — graph tra trượt."""
    out = frame.copy()
    macs = out["mac"] if "mac" in out.columns else pd.Series(MISSING, index=out.index)
    out["mac_oui"] = [mac_oui(v) for v in macs]
    out["mac_is_random"] = [mac_is_random(v) for v in macs]
    for col in ("dhcp_hostname", "mdns_model"):
        out[col] = [_clean_text(v) for v in out[col]] if col in out.columns else MISSING
    return out


def prl_options(value):
    """Tập token chữ số của PRL — bản python của phần `StringSplit` trong `build_derived`."""
    text = str(value)
    if text in NOT_PRESENT:
        return set()
    return {t for t in text.split(PRL_SEP) if re.fullmatch(r"[0-9]+", t)}


def derive_numeric(frame, num_cols):
    """Cột số encoder cần + `SOURCE_FLAGS`, tính lại từ cột chuỗi — khớp `build_derived`.

    "Có giá trị" là so khớp chính xác với `NOT_PRESENT`, không strip, vì graph cũng chỉ so
    `Equal` từng chuỗi.
    """
    out = pd.DataFrame(index=frame.index)
    for flag, sources in SOURCE_EVIDENCE.items():
        hit = np.zeros(len(frame), dtype=bool)
        for col in sources:
            hit |= ~frame[col].astype(str).isin(NOT_PRESENT).to_numpy()
        out[flag] = hit.astype(np.float32)
    options = [prl_options(v) for v in frame["dhcp_prl"]]
    for col in num_cols:
        if col.startswith("dhcp_opt_"):
            code = col.rsplit("_", 1)[1]
            out[col] = np.float32([code in o for o in options])
    out["dhcp_prl_len"] = np.float32([len(o) for o in options])
    unknown = [c for c in num_cols if c not in out.columns]
    if unknown:
        raise KeyError(f"không suy được cột số từ chuỗi: {unknown}")
    return out


def matrix_frame(matrix, cols, num_cols):
    """Ma trận chuỗi graph nhận -> frame đủ cột cho encoder, cột số tự tính như graph."""
    frame = pd.DataFrame(matrix, columns=list(cols)).astype(str)
    return pd.concat([frame, derive_numeric(frame, num_cols)], axis=1)


def to_input_matrix(frame, columns):
    """frame -> object array [N, len(columns)] toàn chuỗi, đúng thứ tự contract."""
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(f"thiếu cột: {missing}")
    out = frame[list(columns)].copy()
    for col in columns:
        v = out[col]
        out[col] = v.astype(np.float32).astype(str) if pd.api.types.is_numeric_dtype(v) else v.astype(str)
    return out.to_numpy(dtype=object)


def encode(frame, encoder):
    return encoder["ct"].transform(prepare_frame(frame, encoder)).astype(np.float32)


OUI_RE = re.compile(r"[0-9a-fA-F]{2}:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}")


def mac_random_from_oui(value):
    """`mac_is_random` suy từ cột `mac_oui` của contract 12 cột — quy ước của head OS trong 12/13.

    Router luôn biết MAC; extractor chỉ gửi `<missing>` khi MAC ngẫu nhiên hoá. OUI gửi nguyên thì xét
    bit U/L (0x02) của octet đầu.
    """
    v = str(value)
    if v in NOT_PRESENT:
        return 1                              # extractor chỉ trả <missing> khi MAC ngẫu nhiên hoá
    return int(bool(OUI_RE.fullmatch(v)) and int(v[:2], 16) & 0x02 != 0)


def capture_frame():
    """Bộ capture tự thu (`Data/features/*_capture.csv`) ở dạng contract 12 cột."""
    feat = DATA / "features"
    sess = pd.read_csv(feat / "session_features_capture.csv")
    dhcp = pd.read_csv(feat / "dhcp_features_capture.csv")
    hostnames = dhcp.dropna(subset=["hostname"]).drop_duplicates("mac").set_index("mac")["hostname"]
    sess["dhcp_hostname"] = sess["mac"].map(hostnames).fillna(MISSING)
    sess["mdns_model"] = MISSING
    return add_l0_columns(sess)

In [9]:
# --- Head OS (05 phần B) ---
import re


OS_CLASSES = ["iOS", "Android", "Windows", "macOS", "Linux", "Embedded"]
OS_THRESHOLD = 0.6          # ngưỡng của rừng (L2); L0/L1 là luật, conf = 1.0


def prl_set(prl):
    text = str(prl)
    return frozenset() if text in NOT_PRESENT else frozenset(int(t) for t in re.findall(r"\d+", text))


HOST_OS = [
    (r"^(iphone|ipad|ipod)\b", "iOS"),
    (r"^(macbook|imac|mac-?mini|mac-?pro|mac-?studio)\b", "macOS"),
    (r"^(android|galaxy|sm-|redmi|xiaomi|poco|mi-|realme|oppo|vivo|oneplus|pixel|huawei|honor|nokia|moto)", "Android"),
    (r"^(desktop|laptop|win)-[a-z0-9]{4,}$", "Windows"),
    (r"^(raspberrypi|ubuntu|debian|fedora|archlinux)\b", "Linux"),
    (r"^(esp[_-]|tuya|shelly|tasmota|wled)", "Embedded"),
]


def l0_os(hostname, vci, mac_random):
    """Chuỗi thiết bị tự khai."""
    h = str(hostname or "").strip().lower()
    if h and h not in ("<missing>", "nan", "none"):
        for pattern, os_ in HOST_OS:
            if re.search(pattern, h):
                return os_, "hostname"
    v = str(vci or "").strip().lower()
    if v.startswith("msft "):
        return "Windows", "vci"
    # android-dhcp-* cũng có ở loa Echo Studio (MAC thật) -> chỉ tin khi MAC ngẫu nhiên.
    if v.startswith("android-dhcp-") and mac_random:
        return "Android", "vci"
    return None, None


def l1_os(signatures, prl, mdns, mac_random, min_jaccard=0.9):
    """Vân tay giao thức: tập option DHCP gần nhất, rồi Apple Continuity qua mDNS."""
    s = prl_set(prl)
    if s:
        best, owners = 0.0, set()
        for key, value in signatures.items():
            j = len(s & key) / len(s | key)
            if j > best + 1e-9:
                best, owners = j, set(value)
            elif abs(j - best) <= 1e-9:
                owners |= value
        if best >= min_jaccard and len(owners) == 1:
            os_ = next(iter(owners))
            # Vân tay thiết bị gia dụng mà MAC ngẫu nhiên hoá: bằng chứng trái nhau, nhường L2.
            if not (os_ == "Embedded" and mac_random):
                return os_, f"prl~{best:.2f}"
    # Handoff/AirDrop: Apple TV/HomePod cũng quảng bá nhưng dùng MAC thật.
    m = str(mdns)
    if mac_random and ("_companion link" in m or "_rdlink" in m):
        return "iOS", "mdns"
    return None, None


def os_decide(bundle, frame, thr=OS_THRESHOLD, use_hostname=True):
    """-> list (nhãn, nguồn, conf) cho từng dòng."""
    X, _, _ = apply_encoder(frame[bundle["cols"]], bundle["encoder"])
    proba = bundle["model"].predict_proba(X)
    classes = bundle["model"].classes_
    out = []
    for i, (_, r) in enumerate(frame.iterrows()):
        rnd = int(r.get("mac_is_random", 0))
        label, src = l0_os(r.get("dhcp_hostname") if use_hostname else None, r.get("dhcp_vci"), rnd)
        if not label:
            label, src = l1_os(bundle["signatures"], r.get("dhcp_prl"), r.get("mdns_tokens"), rnd)
        if label:
            out.append((label, src, 1.0))
            continue
        k = int(proba[i].argmax()); p, conf = classes[k], float(proba[i, k])
        # Vân tay TLS là của TRÌNH DUYỆT (Chrome trên Windows = Chrome trên Linux), thứ tự
        # cipher của Apple xáo theo phiên: cửa sổ chỉ-TLS không đủ để nói một OS đa dụng.
        tls_only = not (r.get("has_dhcp") == 1 or r.get("has_mdns") == 1)
        if conf >= thr and not (p != "Embedded" and tls_only):
            out.append((p, "forest", conf))
        else:
            out.append((UNKNOWN_LABEL, "abstain", conf))
    return out

In [10]:
# --- Head OS (05 phần B) ---
TYPE_OK = {"iOS": {"Smartphone"}, "Android": {"Smartphone"}, "Windows": {"Laptop"},
           "macOS": {"Laptop"}, "Linux": {"Laptop", "Single-board Computer"}}
GENERAL_PURPOSE_TYPES = {"Smartphone", "Laptop", "Single-board Computer"}
MAKE_OK = {"iOS": {"Apple"}, "macOS": {"Apple"}}


def cascade(os_label, mac_random, make, type_, model):
    """-> (os, make, type, model, ghi chú)."""
    notes = []
    if os_label != UNKNOWN_LABEL:
        allowed = TYPE_OK.get(os_label)
        if os_label == "Embedded" and type_ in GENERAL_PURPOSE_TYPES:
            notes.append(f"type {type_} mâu thuẫn Embedded"); type_ = UNKNOWN_LABEL
        elif type_ != UNKNOWN_LABEL and allowed and type_ not in allowed:
            notes.append(f"type {type_} mâu thuẫn {os_label}"); type_ = UNKNOWN_LABEL
        if type_ == UNKNOWN_LABEL:
            if os_label == "iOS" or (os_label == "Android" and mac_random):
                type_ = "Smartphone"; notes.append("type suy từ OS")
            elif os_label in ("Windows", "macOS"):
                type_ = "Laptop"; notes.append("type suy từ OS")
        allowed = MAKE_OK.get(os_label)
        if make != UNKNOWN_LABEL and allowed and make not in allowed:
            notes.append(f"make {make} mâu thuẫn {os_label}"); make = UNKNOWN_LABEL
        if make == "Apple" and os_label not in ("iOS", "macOS"):
            notes.append(f"make Apple mâu thuẫn {os_label}"); make = UNKNOWN_LABEL
        if make == UNKNOWN_LABEL and allowed:
            make = "Apple"; notes.append("make suy từ OS")
    if (make == UNKNOWN_LABEL or type_ == UNKNOWN_LABEL) and model != UNKNOWN_LABEL:
        notes.append("model bỏ vì make/type abstain"); model = UNKNOWN_LABEL
    return os_label, make, type_, model, "; ".join(notes)

In [11]:
# --- Xuất rừng sang mảng phẳng và TFLite (12, 13) ---
import numpy as np


def float32_floor(values):
    """float32 lớn nhất <= values (float64) — giữ nguyên kết quả của phép so x_f32 <= t."""
    out = values.astype(np.float32)
    up = out.astype(np.float64) > values
    out[up] = np.nextafter(out[up], np.float32(-np.inf))
    return out


def flatten_forest(model):
    feature, threshold, left, right, value, roots = [], [], [], [], [], []
    offset = 0
    for est in model.estimators_:
        t = est.tree_
        n = t.node_count
        own = np.arange(n)
        leaf = t.children_left == -1
        # cây con nhận nhãn đã mã hoá 0..C-1 — cột value theo đúng thứ tự `model.classes_`
        assert t.value.shape[1:] == (1, len(model.classes_)), t.value.shape
        feature.append(np.where(leaf, 0, t.feature))
        threshold.append(np.where(leaf, 0.0, float32_floor(t.threshold)))
        left.append(np.where(leaf, own, t.children_left) + offset)
        right.append(np.where(leaf, own, t.children_right) + offset)
        v = t.value[:, 0, :]
        value.append(v / v.sum(axis=1, keepdims=True))
        roots.append(offset)
        offset += n
    return {
        "feature": np.concatenate(feature).astype(np.int32),
        "threshold": np.concatenate(threshold).astype(np.float32),
        "left": np.concatenate(left).astype(np.int32),
        "right": np.concatenate(right).astype(np.int32),
        "value": np.concatenate(value).astype(np.float32),
        "roots": np.asarray(roots, dtype=np.int32),
        "depth": max(est.tree_.max_depth for est in model.estimators_),
    }


IO_NAMES = (IN_FEATURES, OUT_LABELS)   # tên tensor + khoá signature của .tflite — cùng tên với ONNX


def rename_io(model_bytes, names=IO_NAMES):
    """Đặt tên tensor vào/ra và khoá signature thành `input`/`output`.

    Converter tự đặt tên tensor (`serving_default_input:0`, ra thì lấy tên op cuối, vd. `Shape`), không
    chỉnh được qua API. Sửa thẳng flatbuffer: chỉ đổi chuỗi tên, graph không đổi.
    """
    from tensorflow.lite.tools import flatbuffer_utils

    model = flatbuffer_utils.read_model_from_bytearray(bytearray(model_bytes))
    assert len(model.subgraphs) == 1
    sg = model.subgraphs[0]
    assert len(sg.inputs) == 1 and len(sg.outputs) == 1, (sg.inputs, sg.outputs)
    sg.tensors[sg.inputs[0]].name = names[0]
    sg.tensors[sg.outputs[0]].name = names[1]
    for sig in model.signatureDefs or []:
        sig.inputs[0].name, sig.outputs[0].name = names
    return bytes(flatbuffer_utils.convert_object_to_bytearray(model))

CONTRACT_KEY = "sdc.contract"


def embed_metadata(model_bytes, name, payload):
    from tensorflow.lite.tools import flatbuffer_utils

    from tensorflow.lite.python import schema_py_generated as tfl_schema

    model = flatbuffer_utils.read_model_from_bytearray(bytearray(model_bytes))
    names = [m.name.decode() if isinstance(m.name, bytes) else m.name for m in model.metadata or []]
    assert name not in names, f"metadata {name!r} đã có"
    buf = tfl_schema.BufferT()
    buf.data = np.frombuffer(payload, dtype=np.uint8)
    model.buffers.append(buf)
    entry = tfl_schema.MetadataT()
    entry.name, entry.buffer = name, len(model.buffers) - 1
    model.metadata = (model.metadata or []) + [entry]
    return bytes(flatbuffer_utils.convert_object_to_bytearray(model))


def read_metadata(model_bytes, name):
    """Chỉ dùng schema flatbuffer — không cần TF runtime, như code phía thiết bị."""
    from tensorflow.lite.python import schema_py_generated as tfl_schema

    model = tfl_schema.Model.GetRootAsModel(model_bytes, 0)
    for i in range(model.MetadataLength()):
        entry = model.Metadata(i)
        if entry.Name().decode() == name:
            return model.Buffers(entry.Buffer()).DataAsNumpy().tobytes()
    raise KeyError(name)

In [12]:
from pathlib import Path
import hashlib
import inspect
import itertools
import json
import re
import warnings
from datetime import datetime

import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import tensorflow as tf
from IPython.display import display

warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")
ort.set_default_logger_severity(3)
print("TensorFlow", tf.__version__, "| onnxruntime", ort.__version__)

TensorFlow 2.22.0-rc0 | onnxruntime 1.28.0


## 1. Chọn release

`RUN` = tên thư mục run `Models/<ts>/`; `None` = run mới nhất đã có `release/` (12). `train/` và `iden/` lấy từ cùng run, kèm kiểm sha256 theo `manifest.json` của release.

**`mac_is_random` không có trong 12 cột.** Head OS cần nó (luật VCI Android, chữ ký PRL, mDNS Apple, rừng,
cascade). Suy từ `mac_oui`: extractor (`mac_oui()` của 09) chỉ trả `<missing>` khi MAC ngẫu nhiên hoá hoặc hỏng,
mà router luôn biết MAC của thiết bị — nên `<missing>` được hiểu là **MAC ngẫu nhiên**. Caller gửi nguyên OUI
(09 cho phép) thì xét bit U/L `0x02`. Mục 5 kiểm quy ước này khớp `mac_is_random(MAC)` trên mọi dòng thật.

In [13]:
COLS = list(INPUT_COLUMNS)

RUN = None   # run Models/<ts>/ đã qua 12; None = mới nhất

run_dir = pick_run(RUN, f"release/{RELEASE_MANIFEST}")
release_dir, iden_dir, train_dir = run_dir / "release", run_dir / "iden", run_dir / "train"
manifest = json.loads((release_dir / RELEASE_MANIFEST).read_text(encoding="utf-8"))
src = manifest["sources"]
release_path = release_dir / RELEASE_ONNX
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(release_path) == manifest["files"][RELEASE_ONNX]["sha256"], "sdc_os.onnx bị sửa sau khi 12 ghi manifest"

onnx_path = iden_dir / "sdc_iden.onnx"
assert sha(onnx_path) == src["iden_onnx_sha256"], "sdc_iden.onnx khác bản 12 đã ghép"
onnx_model = onnx.load(str(onnx_path))
onnx_contract = json.loads({p.key: p.value for p in onnx_model.metadata_props}["sdc.contract"])
assert onnx_contract["format"] == IDEN_FORMAT, onnx_contract["format"]
assert onnx_contract["columns"] == COLS, "thứ tự cột của ONNX khác contract hiện hành"

bundle = joblib.load(train_dir / "iden.joblib")
enc = bundle["encoder"]
LABELS = {h: [str(c) for c in bundle["models"][h].classes_] for h in HEADS}
assert LABELS == onnx_contract["labels"], "nhãn của bundle khác nhãn trong ONNX"

os_bundle = joblib.load(train_dir / "os.joblib")
assert os_bundle["format"] == "sdc-os-v0", os_bundle["format"]
assert OS_THRESHOLD == os_bundle["threshold"], "ngưỡng OS (mục Thư viện) lệch run đã lưu"
assert list(OS_CLASSES) == list(os_bundle["classes"])
MIN_JACCARD = inspect.signature(l1_os).parameters["min_jaccard"].default
OS_LABELS = list(os_bundle["classes"]) + [UNKNOWN_LABEL]
print(f"run: {run_dir.name}  |  release {manifest['release_id']}")
print(", ".join(f"{h} {len(LABELS[h])} lớp" for h in HEADS), f"|  lớp OS {OS_LABELS}  |  ngưỡng rừng OS {os_bundle['threshold']}")

release: 20261005_140727_release  |  train: 20261005_140503_train  |  iden: 20261005_140605_iden
make 9 lớp, type 5 lớp, model 12 lớp |  lớp OS ['iOS', 'Android', 'Windows', 'macOS', 'Linux', 'Embedded', '__unknown__']  |  ngưỡng rừng OS 0.6


## 2. Trích bảng từ graph ONNX

Tên node do `GB` của 09 sinh nên có tiền tố cố định: `sdc/le/<mode>/<head>_*` (vân tay L1),
`sdc/le/oui_*`, `sdc/le/mdns_*`, `sdc/rx/<head>_*` + `sdc/host/<head>_*` (luật hostname), `sdc/mask/<head>_*`,
`sdc/lab/<head>_*`, `thresholds`. Luật hostname được 09 dựng **ngược** (luật đầu nằm ngoài cùng), nên đảo lại
thứ tự node là ra thứ tự ưu tiên. Kích thước mọi bảng phải khớp `sizes` trong contract ONNX.

In [14]:
def onnx_tables(model):
    attrs = lambda n: {a.name: onnx.helper.get_attribute_value(a) for a in n.attribute}
    inits = {i.name: onnx.numpy_helper.to_array(i) for i in model.graph.initializer}
    stem = lambda name, prefix: name[len(prefix):].rsplit("_", 1)[0]
    l1 = {mode: {} for mode, _, _ in FP_MODES}
    oui, mdns, rx, host = {}, {}, {}, {h: [] for h in HEADS}
    for node in model.graph.node:
        out = node.output[0]
        if node.op_type == "LabelEncoder" and out.startswith("sdc/le/"):
            a = attrs(node)
            assert a["default_int64"] == -1
            keys = [k.decode() for k in a["keys_strings"]]
            book = {} if keys == ["\x00__empty__"] else dict(zip(keys, map(int, a["values_int64s"])))
            name = stem(out, "sdc/le/")
            if name == "oui":
                oui = book
            elif name == "mdns":
                mdns = book
            else:
                mode, head = name.split("/")
                l1[mode][head] = book
        elif node.op_type == "RegexFullMatch" and out.startswith("sdc/rx/"):
            rx[out] = (stem(out, "sdc/rx/"), attrs(node)["pattern"].decode())
        elif node.op_type == "Where" and out.startswith("sdc/host/"):
            head, pattern = rx[node.input[0]]
            host[head].append((pattern, int(inits[node.input[1]].ravel()[0])))
    find = lambda prefix: next(v for k, v in inits.items() if k.startswith(prefix))
    return {
        "l1": l1, "oui": oui, "mdns_model": mdns,
        "hostname": {h: list(reversed(rules)) for h, rules in host.items()},
        "masks": {h: find(f"sdc/mask/{h}_").astype(np.float32) for h in ("make", "type")},
        "labels": {h: [s.decode() if isinstance(s, bytes) else str(s) for s in find(f"sdc/lab/{h}_").ravel()]
                   for h in HEADS},
        "thresholds": inits["thresholds"].astype(np.float32),
    }


T = onnx_tables(onnx_model)
sizes = onnx_contract["sizes"]
assert {f"{m}/{h}": len(b) for m, hs in T["l1"].items() for h, b in hs.items()} == sizes["l1"]
assert (len(T["oui"]), len(T["mdns_model"])) == (sizes["oui"], sizes["mdns_model"])
assert sum(map(len, T["hostname"].values())) == sizes["hostname"]
assert T["labels"] == LABELS
assert np.allclose(T["thresholds"], [onnx_contract["thresholds"][h] for h in HEADS])
for h in ("make", "type"):
    assert T["masks"][h].shape == (len(LABELS[MODEL_HEAD]), len(LABELS[h]))
print("L1:", sizes["l1"])
print(f"OUI {len(T['oui'])}, mDNS model= {len(T['mdns_model'])}, hostname "
      + ", ".join(f"{h} {len(r)}" for h, r in T["hostname"].items()))

L1: {'fp_full/make': 2, 'fp_full/type': 2, 'fp_full/model': 2, 'fp_dhcp/make': 5, 'fp_dhcp/type': 5, 'fp_dhcp/model': 5, 'fp_tls/make': 3, 'fp_tls/type': 3, 'fp_tls/model': 3}
OUI 6, mDNS model= 9, hostname make 7, type 5, model 5


## 3. Graph TF

**Head iden** — đi đúng thứ tự `build_model` / `reference()` của 09:

1. **Cột và cờ** — tách 12 cột; `has_*` theo `SOURCE_EVIDENCE`; `dhcp_opt_k` = có token `k` trong
   `dhcp_prl` tách theo `,`; `dhcp_prl_len` = số token chữ số **không trùng token đứng trước** (tam giác
   dưới của ma trận so cặp, như `Trilu` của 09).
2. **Encoder** — ordinal: so chuỗi với danh sách category, trượt → `-1`; TF-IDF: lowercase, tách token đúng
   `token_pattern`, đếm theo vocab, nhân idf, chuẩn hoá L2 trong từng khối.
3. **Rừng** — 3 head, duyệt cây bằng gather trên mảng của `flatten_forest`.
4. **Gộp** — L0 (mDNS > OUI > hostname) và L1 (`fp_full` → `fp_dhcp` → `fp_tls`, `-2` dừng chuỗi), head
   `model` chạy trước và lái mask hierarchy cho `make`/`type`, rồi ngưỡng theo số nguồn → `__unknown__`.

**Head OS** — đi đúng `os_decide()` (mục Thư viện):

5. **L0** — `HOST_OS` trên hostname đã `strip().lower()` (`re.search` → full-match `.*(…).*`; `\b` của Python
   là Unicode nên đổi thành `[^\p{L}\p{N}_]|$`), rồi VCI `msft ` / `android-dhcp-` (chỉ khi MAC ngẫu nhiên).
6. **L1** — tập option = token của `dhcp_prl` tách theo `,` (dạng chuẩn của contract, giống hệt `sdc_os.onnx`), Jaccard với từng chữ ký,
   tập OS của các chữ ký đạt max; trả lời khi ≥ `MIN_JACCARD` và đúng một OS (Embedded + MAC ngẫu nhiên thì bỏ
   qua). Rồi mDNS `_companion link` / `_rdlink` + MAC ngẫu nhiên → iOS. Tỉ số Jaccard là phân số nhỏ: hai
   phân số bằng nhau cho đúng cùng một float32, nên so bằng và so max trên float32 là chính xác.
7. **L2** — rừng OS; argmax lấy chỉ số đầu khi hoà (như numpy), trả lời khi conf ≥ ngưỡng, trừ khi lớp khác
   Embedded mà cửa sổ không có DHCP lẫn mDNS.
8. **Cascade** — `cascade()` (mục Thư viện) trên chỉ số nhãn: gạt type/make mâu thuẫn OS, điền type/make từ OS,
   make hoặc type `__unknown__` thì model cũng `__unknown__`.

**Nhãn** — một bảng nhãn chung cho 3 head, gather bằng chỉ số `[N,3]` (op builtin; `PACK` chuỗi thì không).
Bảng tra nhỏ (≤ 61 khoá) nên tra bằng so bằng chuỗi với cả danh sách — không cần hashtable.

In [15]:
VCI_PATTERN, SPACE_PATTERN = r"[^:\-_. ]+", r"[^ ]+"


def encoder_spec(encoder, n_features):
    """Mọi thứ graph cần để dựng lại đúng vector của `ColumnTransformer`."""
    ct = encoder["ct"]
    cats = {c: [str(v) for v in vals]
            for c, vals in zip(encoder["cat_cols"], ct.named_transformers_["cat"].categories_)}
    text = {}
    for c in encoder["text_cols"]:
        vec = ct.named_transformers_[f"t_{c}"]
        p = vec.get_params()
        assert (p["analyzer"], p["ngram_range"], p["norm"], p["sublinear_tf"], p["use_idf"]) == ("word", (1, 1), "l2", False, True)
        assert p["lowercase"] and p["preprocessor"] is None and p["tokenizer"] is None and p["stop_words"] is None
        assert vec.token_pattern in (VCI_PATTERN, SPACE_PATTERN), vec.token_pattern
        text[c] = {"vocab": [str(t) for t in vec.get_feature_names_out()],
                   "idf": vec.idf_.astype(np.float32), "split_punct": vec.token_pattern == VCI_PATTERN}
    spec = {"num": list(encoder["num_cols"]), "cat": cats, "text": text, "width": int(n_features)}
    assert len(spec["num"]) + len(cats) + sum(len(t["vocab"]) for t in text.values()) == n_features
    assert set(cats) | set(text) <= set(COLS)
    return spec


SPEC = {"iden": encoder_spec(enc, bundle["models"]["make"].n_features_in_)}
FLAT = {h: flatten_forest(bundle["models"][h]) for h in HEADS}
assert set(SPEC["iden"]["cat"]) | set(SPEC["iden"]["text"]) == set(ENCODER_STRINGS)
SPEC["os"] = encoder_spec(os_bundle["encoder"], os_bundle["model"].n_features_in_)
FLAT["os"] = flatten_forest(os_bundle["model"])
assert set(SPEC["os"]["num"]) <= set(SPEC["iden"]["num"]) | {"mac_is_random"}
for name, s in SPEC.items():
    print(f"encoder {name}: {s['width']} feature = num {len(s['num'])} + cat {len(s['cat'])} + "
          + " + ".join(f"{c} {len(t['vocab'])}" for c, t in s["text"].items()))
print("rừng:", {h: (len(f["roots"]), len(f["feature"]), f["depth"]) for h, f in FLAT.items()})

# --- Hằng của head OS + cascade -------------------------------------------------------------


def search_as_fullmatch(pattern):
    """`re.search` (Python) -> full-match RE2. `\b` cuối mẫu là biên phải Unicode."""
    if r"\b" in pattern:
        assert pattern.endswith(r"\b") and pattern.count(r"\b") == 1, pattern
        pattern = pattern[:-2] + r"(?:[^\p{L}\p{N}_]|$)"
    return f"(?s:.*)(?:{pattern})(?s:.*)"


# Khoảng trắng mà str.strip() của Python bỏ: \t\n\v\f\r, \x1c-\x1f, \x85 và mọi ký tự nhóm Z.
PY_SPACE = r"[\t\n\x{0b}\f\r\x{1c}-\x{1f}\x{85}\p{Z}]"

OSI = {c: i for i, c in enumerate(OS_LABELS)}
HOST_OS_RE = [(search_as_fullmatch(p), OSI[o]) for p, o in HOST_OS]
FOREST_OS = np.int32([OSI[str(c)] for c in os_bundle["model"].classes_])
SIG_KEYS = list(os_bundle["signatures"])
SIG_OPTIONS = sorted({o for k in SIG_KEYS for o in k})
SIG_MEMBER = np.float32([[o in k for k in SIG_KEYS] for o in SIG_OPTIONS])                   # [U, K]
SIG_SIZE = np.float32([len(k) for k in SIG_KEYS])
SIG_OWNER = np.float32([[c in os_bundle["signatures"][k] for c in OS_LABELS] for k in SIG_KEYS])  # [K, O]
TYPE_LABELS, MAKE_LABELS = LABELS["type"], LABELS["make"]
CAS = {
    "type_rule": np.bool_([o in TYPE_OK for o in OS_LABELS]),
    "type_ok": np.float32([[t in TYPE_OK.get(o, ()) for t in TYPE_LABELS] for o in OS_LABELS]),
    "general": np.bool_([t in GENERAL_PURPOSE_TYPES for t in TYPE_LABELS]),
    "make_rule": np.bool_([o in MAKE_OK for o in OS_LABELS]),
    "make_ok": np.float32([[m in MAKE_OK.get(o, ()) for m in MAKE_LABELS] for o in OS_LABELS]),
}


class SdcModule(tf.Module):
    def __init__(self):
        super().__init__()
        spec = [tf.TensorSpec([None, len(COLS)], tf.string, name=IO_NAMES[0])]
        self.run = tf.function(self._run, input_signature=spec)
        self.debug = tf.function(self._core, input_signature=spec)
        self.cascade_fn = tf.function(self.cascade, input_signature=[
            tf.TensorSpec([None], tf.int32), tf.TensorSpec([None], tf.bool)]
            + [tf.TensorSpec([None], tf.int32)] * 3)

    # --- tiện ích ----------------------------------------------------------------------
    @staticmethod
    def present(x):
        bad = tf.equal(tf.expand_dims(x, 1), tf.constant(list(NOT_PRESENT)))
        return tf.logical_not(tf.reduce_any(bad, axis=1))

    @staticmethod
    def lookup(x, book):
        """Chuỗi -> giá trị trong `book`, trượt -> -1 (như LabelEncoder default)."""
        if not book:
            return tf.fill(tf.shape(x), -1)
        eq = tf.equal(tf.expand_dims(x, 1), tf.constant(list(book)))
        pos = tf.argmax(tf.cast(eq, tf.int32), axis=1, output_type=tf.int32)
        return tf.where(tf.reduce_any(eq, axis=1),
                        tf.gather(tf.constant(list(book.values()), tf.int32), pos), -1)

    @staticmethod
    def tokens(x, sep):
        return tf.strings.split(x, sep=sep).to_tensor(default_value="")      # [N, M]

    @staticmethod
    def first_occurrence(tok):
        """[N, M] bool: token này không trùng token nào đứng trước nó."""
        same = tf.cast(tf.equal(tf.expand_dims(tok, 2), tf.expand_dims(tok, 1)), tf.float32)
        earlier = tf.linalg.band_part(same, -1, 0) - tf.linalg.band_part(same, 0, 0)   # j < i
        return tf.equal(tf.reduce_sum(earlier, axis=2), 0.0)

    @staticmethod
    def py_strip_lower(x):
        """`str(x).strip().lower()` của Python."""
        x = tf.strings.regex_replace(x, f"^{PY_SPACE}+|{PY_SPACE}+$", "")
        return tf.strings.lower(x, encoding="utf-8")

    def columns(self, inp):
        return {c: inp[:, i] for i, c in enumerate(COLS)}

    def mac_random(self, oui):
        bit = tf.strings.regex_full_match(oui, "[0-9a-fA-F][2367abefABEF]:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}")
        return tf.logical_or(tf.logical_not(self.present(oui)), bit)

    # --- cờ + encoder -------------------------------------------------------------------
    def derived(self, col):
        out = {}
        for flag, sources in SOURCE_EVIDENCE.items():
            hit = self.present(col[sources[0]])
            for c in sources[1:]:
                hit = tf.logical_or(hit, self.present(col[c]))
            out[flag] = tf.cast(hit, tf.float32)
        tok = self.tokens(col["dhcp_prl"], PRL_SEP)
        for c in SPEC["iden"]["num"]:
            if c.startswith("dhcp_opt_"):
                code = c.rsplit("_", 1)[1]
                out[c] = tf.cast(tf.reduce_any(tf.equal(tok, code), axis=1), tf.float32)
        digit = tf.strings.regex_full_match(tok, "[0-9]+")
        out["dhcp_prl_len"] = tf.reduce_sum(
            tf.cast(tf.logical_and(digit, self.first_occurrence(tok)), tf.float32), axis=1)
        out["mac_is_random"] = tf.cast(self.mac_random(col["mac_oui"]), tf.float32)
        return out

    def tfidf(self, x, spec):
        x = tf.strings.lower(x, encoding="utf-8")
        if spec["split_punct"]:
            x = tf.strings.regex_replace(x, r"[:\-_.]", " ")
        tok = self.tokens(x, " ")
        counts = tf.reduce_sum(tf.cast(tf.equal(tf.expand_dims(tok, 2), tf.constant(spec["vocab"])),
                                       tf.float32), axis=1)                         # [N, V]
        v = counts * tf.constant(spec["idf"])
        norm = tf.sqrt(tf.reduce_sum(v * v, axis=1, keepdims=True))
        return tf.math.divide_no_nan(v, norm)

    def encode(self, col, derived, spec):
        parts = [tf.stack([derived[c] for c in spec["num"]], axis=1)]
        parts.append(tf.stack([tf.cast(self.lookup(col[c], dict(zip(cats, range(len(cats))))), tf.float32)
                               for c, cats in spec["cat"].items()], axis=1))
        parts += [self.tfidf(col[c], t) for c, t in spec["text"].items()]
        return tf.concat(parts, axis=1)

    @staticmethod
    def forest(x, f, width):
        n = tf.shape(x)[0]
        flat_x = tf.reshape(x, [-1])
        row = tf.expand_dims(tf.range(n) * width, 1)
        node = tf.tile(tf.expand_dims(tf.constant(f["roots"]), 0), tf.stack([n, 1]))
        feature, threshold = tf.constant(f["feature"]), tf.constant(f["threshold"])
        left, right = tf.constant(f["left"]), tf.constant(f["right"])
        for _ in range(int(f["depth"])):
            xv = tf.gather(flat_x, tf.gather(feature, node) + row)
            node = tf.where(xv <= tf.gather(threshold, node), tf.gather(left, node), tf.gather(right, node))
        return tf.reduce_mean(tf.gather(tf.constant(f["value"]), node), axis=1)

    # --- head iden: L0/L1 + rừng + hierarchy + ngưỡng -------------------------------------
    def l1(self, col, flags, head):
        answer, decided = None, None
        for mode, cols, mode_flags in FP_MODES:
            parts = []
            for c in cols:
                parts += [col[c], tf.fill(tf.shape(col[c]), KEY_SEP)]
            key = tf.strings.join(parts[:-1]) if len(cols) > 1 else col[cols[0]]
            raw = self.lookup(key, T["l1"][mode].get(head, {}))
            usable = tf.ones_like(raw, dtype=tf.bool)
            for flag in mode_flags:
                usable = tf.logical_and(usable, tf.equal(flags[flag], 1.0))
            for c in cols:
                usable = tf.logical_and(usable, self.present(col[c]))
            raw = tf.where(usable, raw, -1)
            fresh = tf.where(raw >= 0, raw, -1)
            if answer is None:
                answer, decided = fresh, tf.not_equal(raw, -1)
            else:
                answer = tf.where(decided, answer, fresh)
                decided = tf.logical_or(decided, tf.not_equal(raw, -1))
        return answer

    def l0(self, col, head):
        out = []
        if head == MODEL_HEAD and T["mdns_model"]:
            out.append(self.lookup(col["mdns_model"], T["mdns_model"]))
        if head == "make" and T["oui"]:
            out.append(self.lookup(col["mac_oui"], T["oui"]))
        host = None
        for pattern, idx in reversed(T["hostname"][head]):
            match = tf.strings.regex_full_match(col["dhcp_hostname"], pattern)
            host = tf.where(match, idx, host if host is not None else tf.fill(tf.shape(match), -1))
        if host is not None:
            out.append(host)
        return out

    def fuse(self, head, p, col, flags, n_sources, allow=None):
        n_class = len(LABELS[head])

        def gate(v):
            if allow is None:
                return v
            ok = tf.reduce_sum(allow * tf.one_hot(tf.maximum(v, 0), n_class), axis=1) > 0
            return tf.where(ok, v, -1)

        gated = [gate(c) for c in self.l0(col, head)]
        override = gate(self.l1(col, flags, head))
        for c in reversed(gated):
            override = tf.where(c >= 0, c, override)
        has_override = override >= 0
        tie = tf.constant(np.arange(n_class, dtype=np.float32) * np.float32(TIEBREAK))
        forest_idx = tf.argmax(p - tie, axis=1, output_type=tf.int32)
        conf = tf.where(has_override, 1.0, tf.reduce_max(p, axis=1))
        index = tf.where(has_override, override, forest_idx)
        thr = tf.gather(tf.constant(T["thresholds"][HEADS.index(head)]), n_sources)
        return tf.where(conf < thr, LABELS[head].index(ABSTAIN_LABEL), index)

    # --- head OS: os_decide() (mục Thư viện) ----------------------------------------------
    def os_head(self, col, flags, x_os, random):
        unk = OSI[UNKNOWN_LABEL]
        h = self.py_strip_lower(col["dhcp_hostname"])
        h_ok = tf.logical_not(tf.reduce_any(
            tf.equal(tf.expand_dims(h, 1), tf.constant(["", MISSING, "nan", "none"])), axis=1))
        host = tf.fill(tf.shape(h), -1)
        for pattern, idx in reversed(HOST_OS_RE):            # luật đầu tiên khớp thắng
            host = tf.where(tf.strings.regex_full_match(h, pattern), idx, host)
        host = tf.where(h_ok, host, -1)

        v = self.py_strip_lower(col["dhcp_vci"])
        android = tf.logical_and(tf.strings.regex_full_match(v, "(?s)android-dhcp-.*"), random)
        vci = tf.where(tf.strings.regex_full_match(v, "(?s)msft .*"), OSI["Windows"],
                       tf.where(android, OSI["Android"], -1))

        # L1 PRL — token `dhcp_prl` tách theo `,` (dạng chuẩn của contract), y như sdc_os.onnx;
        # |tập option| = `dhcp_prl_len` đã tính ở `derived`
        tok = self.tokens(col["dhcp_prl"], PRL_SEP)
        n_opt = flags["dhcp_prl_len"]
        has = tf.reduce_any(tf.equal(tf.expand_dims(tok, 2),
                                     tf.constant([str(o) for o in SIG_OPTIONS])), axis=1)  # [N, U]
        inter = tf.matmul(tf.cast(has, tf.float32), tf.constant(SIG_MEMBER))             # [N, K]
        ratio = inter / (tf.expand_dims(n_opt, 1) + tf.constant(SIG_SIZE) - inter)
        best = tf.reduce_max(ratio, axis=1)
        tied = tf.cast(tf.equal(ratio, tf.expand_dims(best, 1)), tf.float32)
        owners = tf.reduce_max(tf.expand_dims(tied, 2) * tf.constant(SIG_OWNER), axis=1)  # [N, O]
        prl_os = tf.argmax(owners, axis=1, output_type=tf.int32)
        prl_ok = (n_opt > 0) & (best >= MIN_JACCARD) & tf.equal(tf.reduce_sum(owners, axis=1), 1.0)
        prl_ok = prl_ok & tf.logical_not(tf.equal(prl_os, OSI["Embedded"]) & random)

        mdns = random & tf.strings.regex_full_match(col["mdns_tokens"], "(?s).*(?:_companion link|_rdlink).*")

        # L2 rừng — argmax lấy chỉ số đầu khi hoà, như numpy
        p = self.forest(x_os, FLAT["os"], SPEC["os"]["width"])
        pmax = tf.reduce_max(p, axis=1)
        n_class = len(FOREST_OS)
        k = tf.reduce_min(tf.where(tf.equal(p, tf.expand_dims(pmax, 1)), tf.range(n_class), n_class), axis=1)
        forest_os = tf.gather(tf.constant(FOREST_OS), k)
        tls_only = tf.logical_not(tf.equal(flags["has_dhcp"], 1.0) | tf.equal(flags["has_mdns"], 1.0))
        forest_ok = (pmax >= float(os_bundle["threshold"])) & tf.logical_not(
            tf.not_equal(forest_os, OSI["Embedded"]) & tls_only)

        os_idx = tf.where(forest_ok, forest_os, unk)
        os_idx = tf.where(mdns, OSI["iOS"], os_idx)
        os_idx = tf.where(prl_ok, prl_os, os_idx)
        os_idx = tf.where(vci >= 0, vci, os_idx)
        return tf.where(host >= 0, host, os_idx)

    def cascade(self, os_idx, random, make, type_, model):
        """`cascade()` (mục Thư viện) trên chỉ số nhãn."""
        t_unk, m_unk, mo_unk = (LABELS[h].index(UNKNOWN_LABEL) for h in ("type", "make", MODEL_HEAD))
        known = tf.not_equal(os_idx, OSI[UNKNOWN_LABEL])
        is_os = lambda *names: tf.reduce_any(tf.stack([tf.equal(os_idx, OSI[n]) for n in names]), axis=0)
        pick = lambda table, idx, n: tf.reduce_sum(tf.gather(tf.constant(table), os_idx) * tf.one_hot(idx, n), axis=1) > 0

        emb_general = is_os("Embedded") & tf.gather(tf.constant(CAS["general"]), type_)
        conflict = (tf.not_equal(type_, t_unk) & tf.gather(tf.constant(CAS["type_rule"]), os_idx)
                    & tf.logical_not(pick(CAS["type_ok"], type_, len(TYPE_LABELS))))
        type_ = tf.where(known & (emb_general | conflict), t_unk, type_)
        empty = known & tf.equal(type_, t_unk)
        phone = empty & (is_os("iOS") | (is_os("Android") & random))
        laptop = empty & tf.logical_not(phone) & is_os("Windows", "macOS")
        type_ = tf.where(phone, TYPE_LABELS.index("Smartphone"), tf.where(laptop, TYPE_LABELS.index("Laptop"), type_))

        make_rule = tf.gather(tf.constant(CAS["make_rule"]), os_idx)
        bad = tf.not_equal(make, m_unk) & make_rule & tf.logical_not(pick(CAS["make_ok"], make, len(MAKE_LABELS)))
        make = tf.where(known & bad, m_unk, make)
        apple = MAKE_LABELS.index("Apple")
        make = tf.where(known & tf.equal(make, apple) & tf.logical_not(is_os("iOS", "macOS")), m_unk, make)
        make = tf.where(known & tf.equal(make, m_unk) & make_rule, apple, make)

        drop = (tf.equal(make, m_unk) | tf.equal(type_, t_unk)) & tf.not_equal(model, mo_unk)
        return make, type_, tf.where(drop, mo_unk, model)

    # --- lắp ráp ------------------------------------------------------------------------
    def _core(self, input):
        col = self.columns(input)
        flags = self.derived(col)
        x = self.encode(col, flags, SPEC["iden"])
        n_sources = tf.cast(tf.add_n([flags[f] for f in SOURCE_FLAGS]), tf.int32)
        proba = {h: self.forest(x, FLAT[h], SPEC["iden"]["width"]) for h in HEADS}
        idx = {MODEL_HEAD: self.fuse(MODEL_HEAD, proba[MODEL_HEAD], col, flags, n_sources)}
        for h in ("make", "type"):
            row = tf.gather(tf.constant(T["masks"][h]), idx[MODEL_HEAD])
            idx[h] = self.fuse(h, proba[h] * row, col, flags, n_sources, allow=row)
        out = {"x_iden": x, "iden": tf.stack([idx[h] for h in HEADS], axis=1)}
        random = tf.equal(flags["mac_is_random"], 1.0)
        x_os = self.encode(col, flags, SPEC["os"])
        os_idx = self.os_head(col, flags, x_os, random)
        idx["make"], idx["type"], idx[MODEL_HEAD] = self.cascade(
            os_idx, random, idx["make"], idx["type"], idx[MODEL_HEAD])
        out.update(x_os=x_os, os=os_idx)
        out["final"] = tf.stack([idx[h] for h in HEADS], axis=1)
        return out

    def _run(self, input):
        offset = np.cumsum([0] + [len(LABELS[h]) for h in HEADS])[:-1]
        table = tf.constant([lab for h in HEADS for lab in LABELS[h]])
        return {IO_NAMES[1]: tf.gather(table, self._core(input)["final"] + tf.constant(offset, tf.int32))}


module = SdcModule()

encoder iden: 1088 feature = num 31 + cat 5 + dhcp_vci 32 + dns_tokens 575 + mdns_tokens 118 + tls_sni_tokens 327
encoder os: 187 feature = num 32 + cat 5 + dhcp_vci 32 + mdns_tokens 118
rừng: {'make': (250, 16570, 31), 'type': (250, 13196, 24), 'model': (250, 19486, 30), 'os': (300, 13052, 20)}


## 4. Convert

`TFLITE_BUILTINS` trước, `SELECT_TF_OPS` cho phần còn lại. Danh sách op Flex in ra dưới đây chính là thứ
runtime trên thiết bị phải có.

In [16]:
converter = tf.lite.TFLiteConverter.from_concrete_functions([module.run.get_concrete_function()], module)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite_bytes = rename_io(converter.convert())


def make_interpreter(model_bytes):
    # tensorflow <= 2.16 trên Linux tự gắn Flex delegate; từ 2.20 (mọi OS) và ai_edge_litert thì không.
    return tf.lite.Interpreter(model_content=model_bytes)


_it = make_interpreter(tflite_bytes)
try:                       # op Flex chỉ lỗi lúc invoke, nên chạy thử hẳn một dòng
    _probe = make_interpreter(tflite_bytes)
    _probe.resize_tensor_input(0, [1, len(COLS)])
    _probe.allocate_tensors()
    _probe.set_tensor(_probe.get_input_details()[0]["index"],
                      np.array([[MISSING.encode()] * len(COLS)], dtype=object))
    _probe.invoke()
    FLEX_OK = True
except RuntimeError as exc:
    FLEX_OK = False
    print("Interpreter ở máy này KHÔNG có Flex delegate — mục 5 đối chiếu bằng graph TF eager, "
          "file .tflite kiểm lại trên máy có Flex bằng parity_inputs.npz (mục 6).")
OPS = sorted({op["op_name"] for op in _it._get_ops_details()})
FLEX_OPS = [op for op in OPS if op.startswith("Flex")]
io = (_it.get_input_details()[0], _it.get_output_details()[0])
assert (io[0]["name"], io[1]["name"]) == IO_NAMES
print(f"TFLite {len(tflite_bytes) / 1024:.0f} KiB | input {io[0]['name']} {io[0]['dtype'].__name__} "
      f"{list(io[0]['shape_signature'])} | output {io[1]['name']} {list(io[1]['shape_signature'])}")
print("builtin:", [op for op in OPS if not op.startswith("Flex")])
print("Flex   :", FLEX_OPS)

INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpbwhlmhp0\assets


INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpbwhlmhp0\assets


Interpreter ở máy này KHÔNG có Flex delegate — mục 5 đối chiếu bằng graph TF eager, file .tflite kiểm lại trên máy có Flex bằng parity_inputs.npz (mục 6).
TFLite 3244 KiB | input input bytes_ [np.int32(-1), np.int32(12)] | output output [np.int32(-1), np.int32(3)]
builtin: ['ADD', 'ADD_N', 'ARG_MAX', 'CAST', 'CONCATENATION', 'DIV', 'EQUAL', 'EXPAND_DIMS', 'FILL', 'FULLY_CONNECTED', 'GATHER', 'GREATER', 'GREATER_EQUAL', 'LESS', 'LESS_EQUAL', 'LOGICAL_AND', 'LOGICAL_NOT', 'LOGICAL_OR', 'MAXIMUM', 'MEAN', 'MUL', 'NEG', 'NOT_EQUAL', 'ONE_HOT', 'PACK', 'RANGE', 'REDUCE_ANY', 'REDUCE_MAX', 'REDUCE_MIN', 'RESHAPE', 'SELECT_V2', 'SHAPE', 'SQRT', 'STRIDED_SLICE', 'SUB', 'SUM', 'TILE', 'ZEROS_LIKE']
Flex   : ['FlexRaggedTensorToTensor', 'FlexStaticRegexFullMatch', 'FlexStaticRegexReplace', 'FlexStringJoin', 'FlexStringLower', 'FlexStringSplitV2']


## 5. Kiểm tra

**Chuẩn so là `sdc_os.onnx` của release** — 12 đã kiểm file đó với đường Python (`os_decide`, `cascade`). Luôn chạy
(vài chục giây):

* cascade trên **mọi** tổ hợp đầu vào, so với `cascade()`;
* bộ pcap tự thu (`dhcp_hostname` thật);
* **dòng ép L0 iden** — hostname khớp từng luật, mDNS `model=`, OUI trong bảng, OUI có bit ngẫu nhiên;
* **dòng ép OS** — mọi luật `HOST_OS` (hoa/thường, khoảng trắng, chữ Việt sát biên `\b`), VCI, biến thể PRL quanh
  từng chữ ký, mDNS Apple — mỗi thứ với MAC thiếu / thật / ngẫu nhiên.

`VERIFY = True` thêm `sessions_verified` (toàn bộ dữ liệu train) và **fuzz**, và so từng tầng bên trong: vector
feature với `ColumnTransformer` (`Δx`), nhãn head iden với `sdc_iden.onnx`, OS với `os_decide()` (chỉ trên dòng có `dhcp_prl` dạng chuẩn — `os_decide` đọc PRL kiểu `int()`, còn graph theo contract).

| Cột | So cái gì | Chạy khi |
|---|---|---|
| `<head> lệch (TF)` | output graph TF với `sdc_os.onnx` | luôn |
| `<head> lệch (TFLite)` | output **file `.tflite`** với `sdc_os.onnx` | interpreter có Flex delegate |
| `Δx iden` / `Δx OS`, `iden lệch`, `os lệch` | từng tầng với đường Python | `VERIFY = True` |

Op Flex trong file chính là kernel TensorFlow, nên cột TF đã kiểm cùng một đoạn code tính toán; cột TFLite
kiểm thêm phần chuyển đổi và runtime. Gói `tensorflow` từ 2.20 (Windows lẫn Linux) **không còn** gắn Flex
delegate vào `tf.lite.Interpreter`; `tensorflow==2.16.2` trên Linux (Python 3.12) thì còn. Không có Flex thì mục
6 ghi `parity_inputs.npz` để chạy cột TFLite ở máy có:

```python
import numpy as np, tensorflow as tf                      # tensorflow==2.16.2 trên Linux: có sẵn Flex
d = np.load("parity_inputs.npz", allow_pickle=True)
it = tf.lite.Interpreter(model_path="sdc_os.tflite")
for name in d["names"]:
    m, want = d[f"{name}/input"], d[f"{name}/expected"]
    it.resize_tensor_input(0, m.shape); it.allocate_tensors()
    it.set_tensor(it.get_input_details()[0]["index"], np.vectorize(str.encode, otypes=[object])(m))
    it.invoke()
    got = it.get_tensor(it.get_output_details()[0]["index"]).astype(str)
    print(name, (got != want).sum(axis=0))                 # phải là [0 0 0]
```

In [17]:
VERIFY = False   # True = thêm sessions_verified + fuzz và so từng tầng với đường Python (vài phút)

# Cascade: mọi tổ hợp (OS, MAC ngẫu nhiên, make, type, model) — graph so với cascade()
combos = list(itertools.product(range(len(OS_LABELS)), (False, True), range(len(LABELS["make"])),
                                range(len(LABELS["type"])), range(len(LABELS[MODEL_HEAD]))))
o, r, mk, ty, mo = (np.asarray(v) for v in zip(*combos))
got = [t.numpy() for t in module.cascade_fn(tf.constant(o, tf.int32), tf.constant(r), tf.constant(mk, tf.int32),
                                            tf.constant(ty, tf.int32), tf.constant(mo, tf.int32))]
bad = 0
for i in range(len(combos)):
    want = cascade(OS_LABELS[o[i]], int(r[i]), LABELS["make"][mk[i]], LABELS["type"][ty[i]],
                            LABELS[MODEL_HEAD][mo[i]])[1:4]
    bad += want != (LABELS["make"][got[0][i]], LABELS["type"][got[1][i]], LABELS[MODEL_HEAD][got[2][i]])
print(f"cascade: {len(combos)} tổ hợp, lệch {bad}")
assert bad == 0

cascade: 7560 tổ hợp, lệch 0


In [18]:
def as_matrix(rows):
    return np.asarray([[str(r[c]) for c in COLS] for r in rows], dtype=object)


def iden_probe(base):
    """Dòng ép L0 của iden: mọi luật hostname x (mDNS model=) x (OUI) trên vài dòng nền thật."""
    hostnames = ["Johns-iPhone", "iphone", "DESKTOP-AB12CD3", "desktop-ducanh", "LAPTOP-VAF70SQ6",
                 "Redmi-Note-10", "redmi note 14 pro", "OPPO-A92", "oppo_a5", "raspberrypi", "raspberrypi4-64",
                 "android-0123456789abcdef", "Galaxy-A51", "samsung-cua-duc-anh", "MacBook-Pro", "Thu-iPad",
                 "poco_f3", "xiaomi 13", "linova", MISSING]
    mdns = [MISSING, *list(T["mdns_model"])[:3], "iPhone99,9"]
    ouis = [MISSING, *T["oui"]]
    ouis += [f"{int(o[:2], 16) | 0x02:02x}{o[2:]}" for o in list(T["oui"])[:2]]   # bit U/L bật
    return as_matrix(dict(b, dhcp_hostname=h, mdns_model=m, mac_oui=o)
                     for b in base for h in hostnames for m in mdns for o in ouis)


OS_PROBE = {
    "dhcp_hostname": ["iPhone", " iPhone ", "IPHONE-cua-Lan", "iphoneđ", "iphone-đ", "ipad-pro", "ipod", "MacBook-Air",
                 "imac", "mac-mini", "macpro", "android-0123456789abcdef", "Galaxy-A51", "sm-a515f", "redmi-9",
                 "xiaomi", "poco-x3", "mi-9", "realme-6i", "oppo-a92", "vivo-1906", "oneplus", "pixel-7",
                 "huawei-p30", "honor-x8", "nokia", "moto-g", "DESKTOP-AB12CD3", "desktop-ducanh", "laptop-vaf70sq6",
                 "win-abcd1234", "WIN-AB", "raspberrypi", "raspberrypi4", "ubuntu", "debian-x", "fedora",
                 "archlinux", "ESP_0C3994", "tuya-plug", "shelly1", "tasmota", "wled-1", "linova", "Johns-iPhone",
                 "ĐIỆN-THOẠI", "none", "NaN", "", MISSING],
    "dhcp_vci": ["MSFT 5.0", " msft 5.0", "MSFT5.0", "android-dhcp-11", "ANDROID-DHCP-16", "android-dhcp",
                 "dhcpcd-6.8.2:Linux-4.4.22+:armv7l", "udhcp 1.26.2", MISSING],
    "dhcp_prl": [],
    "mdns_tokens": ["_companion link _tcp local", "_rdlink _tcp local", "x _companion link",
                    "_airplay _tcp local", MISSING],
}
for key in os_bundle["signatures"]:
    opts = [str(v) for v in sorted(key)]
    OS_PROBE["dhcp_prl"] += [",".join(opts), ",".join(reversed(opts)), ",".join(opts[:-1]),
                             ",".join(opts + ["162"]), ",".join([opts[0]] + opts), "0" + ",".join(opts)]


def os_probe(base):
    """Dòng ép từng tầng của head OS, mỗi tầng với MAC thiếu / thật / ngẫu nhiên (OUI thô) / lạ."""
    real = next(iter(T["oui"]))
    ouis = [MISSING, real, f"{int(real[:2], 16) | 0x02:02x}{real[2:]}", "00:11:22", "AA:BB:CC"]
    rows = []
    for b, o in itertools.product(base, ouis):
        rows += [dict(b, dhcp_hostname=h, mac_oui=o) for h in OS_PROBE["dhcp_hostname"]]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=v, mac_oui=o) for v in OS_PROBE["dhcp_vci"]]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=p, mac_oui=o)
                 for p in OS_PROBE["dhcp_prl"]]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=MISSING, mdns_tokens=m, mac_oui=o)
                 for m in OS_PROBE["mdns_tokens"]]
    return as_matrix(rows)


def fuzz_matrix(pools, n, seed=0):
    rng = np.random.default_rng(seed)
    junk = ["", "nan", "None", "x", "1,3,6", "0303|1|h2", "ZZZ unknown token", "1,,3", "01,3"]
    rows = [[str(rng.choice(pools[c] + junk)) for c in COLS] for _ in range(n)]
    for mode, cols, _ in FP_MODES:           # dòng trúng khoá L1, kể cả khoá nhập nhằng
        for head in HEADS:
            for key in T["l1"][mode].get(head, {}):
                r = [str(rng.choice(pools[c])) for c in COLS]
                for c, part in zip(cols, key.split(KEY_SEP)):
                    r[COLS.index(c)] = part
                rows.append(r)
    return np.asarray(rows, dtype=object)


capture = capture_frame()
real_frames = {"capture": capture}
base = capture.drop_duplicates("mac").head(3)[COLS].to_dict("records")
MATRICES = {"capture": to_input_matrix(capture, COLS), "ép L0 iden": iden_probe(base), "ép OS": os_probe(base)}
if VERIFY:
    sessions = add_l0_columns(load_sessions(SESSIONS_PATH)[0])
    real_frames["sessions_verified"] = sessions
    real = pd.concat([sessions[COLS], capture[COLS]], ignore_index=True).astype(str)
    pools = {c: sorted(set(real[c])) for c in COLS}
    pools["mac_oui"] += list(T["oui"])
    pools["mdns_model"] += list(T["mdns_model"])
    for c, values in OS_PROBE.items():
        pools[c] += values
    MATRICES = {"sessions_verified": to_input_matrix(sessions, COLS), **MATRICES,
                "fuzz": fuzz_matrix(pools, 3000)}
print({k: m.shape for k, m in MATRICES.items()})

# Quy ước mac_oui -> mac_is_random phải khớp mac_is_random(MAC) trên mọi dòng thật
for name, frame in real_frames.items():
    rule = np.array([mac_random_from_oui(v) for v in frame["mac_oui"]])
    truth = np.array([mac_is_random(m) for m in frame["mac"]])
    print(f"mac_is_random từ mac_oui — {name}: lệch {int((rule != truth).sum())}/{len(frame)} "
          f"(ngẫu nhiên {int(truth.sum())})")
    assert (rule == truth).all()

{'capture': (62, 12), 'ép L0 iden': (2700, 12), 'ép OS': (2760, 12)}
mac_is_random từ mac_oui — capture: lệch 0/62 (ngẫu nhiên 25)


In [19]:
BATCH = 256
release_sess = ort.InferenceSession(str(release_path), providers=["CPUExecutionProvider"])
iden_sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])


def run_release(m):
    return release_sess.run([OUT_LABELS], {IN_FEATURES: m})[0].astype(str)


def run_tflite(m, model_bytes=None):
    it = make_interpreter(model_bytes or tflite_bytes)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    res = []
    for s in range(0, len(m), BATCH):
        chunk = np.asarray([[v.encode("utf-8") for v in r] for r in m[s:s + BATCH]], dtype=object)
        it.resize_tensor_input(inp["index"], list(chunk.shape))
        it.allocate_tensors()
        it.set_tensor(inp["index"], chunk)
        it.invoke()
        res.append(np.vectorize(lambda b: b.decode("utf-8"))(it.get_tensor(out["index"])))
    return np.concatenate(res)


def run_debug(m):
    parts = [module.debug(tf.constant(m[s:s + BATCH].astype(str))) for s in range(0, len(m), BATCH)]
    return {k: np.concatenate([p[k].numpy() for p in parts]) for k in parts[0]}


def labels_of(idx):
    return np.stack([np.asarray(LABELS[h])[idx[:, i]] for i, h in enumerate(HEADS)], axis=1)


EXPECTED, rows = {}, []
for name, m in MATRICES.items():
    EXPECTED[name] = want = run_release(m)
    dbg = run_debug(m)
    runs = {"TF": labels_of(dbg["final"])}
    if FLEX_OK:
        runs["TFLite"] = run_tflite(m)
    row = {"nguồn": name, "dòng": len(m)}
    for kind, got in runs.items():
        for i, h in enumerate(HEADS):
            row[f"{h} lệch ({kind})"] = int((want[:, i] != got[:, i]).sum())
    if VERIFY:   # từng tầng bên trong, so với đường Python
        frame = matrix_frame(m, COLS, SPEC["iden"]["num"])
        frame["mac_is_random"] = [mac_random_from_oui(v) for v in frame["mac_oui"]]
        row["Δx iden"] = float(np.abs(dbg["x_iden"] - encode(frame, enc)).max())
        row["iden lệch"] = int((labels_of(dbg["iden"]) != iden_sess.run([OUT_LABELS], {IN_FEATURES: m})[0].astype(str)).sum())
        row["Δx OS"] = float(np.abs(dbg["x_os"] - apply_encoder(frame[os_bundle["cols"]], os_bundle["encoder"])[0]).max())
        # os_decide đọc PRL kiểu int() (bỏ số 0 đầu); graph theo dạng chuẩn của contract -> chỉ so dòng PRL chuẩn
        canon = frame["dhcp_prl"].isin(NOT_PRESENT) | frame["dhcp_prl"].str.fullmatch(r"[1-9][0-9]*(,[1-9][0-9]*)*")
        os_py = np.asarray([d[0] for d in os_decide(os_bundle, frame)])
        row["os lệch"] = int(((np.asarray(OS_LABELS)[dbg["os"]] != os_py) & canon.to_numpy()).sum())
        row["PRL ngoài chuẩn (bỏ qua os)"] = int((~canon).sum())
    row["ô đổi bởi cascade"] = int((labels_of(dbg["iden"]) != want).sum())
    rows.append(row)
parity = pd.DataFrame(rows).set_index("nguồn")
display(parity)
assert parity.filter(like="lệch").to_numpy().sum() == 0, "graph TF / file .tflite lệch sdc_os.onnx"
print("KHỚP sdc_os.onnx từng nhãn —", "file .tflite" if FLEX_OK else
      "graph TF (file .tflite: chạy parity_inputs.npz trên máy có Flex, xem trên)")

,dòng,make lệch (TF),type lệch (TF),model lệch (TF),ô đổi bởi cascade
nguồn,,,,,
capture,62,0,0,0,3
ép L0 iden,2700,0,0,0,2442
ép OS,2760,0,0,0,1616


KHỚP sdc_os.onnx từng nhãn — graph TF (file .tflite: chạy parity_inputs.npz trên máy có Flex, xem trên)


In [20]:
# Mẫu output: các cửa sổ của bộ capture, trước và sau cascade
m = MATRICES["capture"]
dbg = run_debug(m)
sample = pd.DataFrame({"dhcp_hostname": m[:, COLS.index("dhcp_hostname")],
                       "mac ngẫu nhiên": [mac_random_from_oui(v) for v in m[:, COLS.index("mac_oui")]],
                       "os": np.asarray(OS_LABELS)[dbg["os"]],
                       "iden (make/type/model)": ["/".join(r) for r in labels_of(dbg["iden"])]})
for i, h in enumerate(HEADS):
    sample[h] = labels_of(dbg["final"])[:, i]
display(sample.drop_duplicates().head(20))

,dhcp_hostname,mac ngẫu nhiên,os,iden (make/type/model),make,type,model
0,lee_kingdom,0,__unknown__,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
1,iphone,1,iOS,Apple/Smartphone/iPhone,Apple,Smartphone,iPhone
2,linova,0,__unknown__,Linova/Linux/Laptop/Linova Laptop HP,Linova/Linux,Laptop,Linova Laptop HP
3,desktop-ducanh,0,Windows,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
4,<missing>,0,__unknown__,Generic Laptop/Laptop/Windows Laptop HP,Generic Laptop,Laptop,Windows Laptop HP
5,desktop-dngjhrt,0,Windows,Generic Laptop/Laptop/Windows Desktop HP,Generic Laptop,Laptop,Windows Desktop HP
6,<missing>,0,Embedded,Camera/IP Camera/Generic IP Camera,Camera,IP Camera,Generic IP Camera
7,<missing>,1,Android,Xiaomi/Smartphone/Redmi Note 10,Xiaomi,Smartphone,Redmi Note 10
8,samsung-cua-duc-anh,1,Android,Samsung/Smartphone/Samsung Galaxy,Samsung,Smartphone,Samsung Galaxy
9,raspberrypi,0,Linux,Raspberry Pi/Single-board Computer/Raspberry Pi,Raspberry Pi,Single-board Computer,Raspberry Pi


## 6. Nhúng contract, ghi vào release

Contract lấy nguyên từ `sdc_os.onnx` của release (cột, nhãn, hash, hierarchy, ngưỡng, `os_head`), thêm phần riêng TFLite: input/output, danh sách op Flex, sha256 của file ONNX nguồn. Ngưỡng **không** ghi đè được lúc chạy như bản ONNX — đổi ngưỡng là chạy lại 09 → 12 → 13.

**Phía thiết bị:** cần LiteRT build kèm Flex delegate (`tensorflow/lite/delegates/flex`, hoặc AAR
`tensorflow-lite-select-tf-ops` trên Android). Trên Linux/ARM phải tự build `libtensorflowlite_flex.so` —
nặng hơn onnxruntime rất nhiều; với router OpenWrt nên dùng `sdc_os.onnx`.

In [21]:
release_contract = json.loads({p.key: p.value for p in onnx.load(str(release_path)).metadata_props}["sdc.contract"])
contract = dict(release_contract)
contract.update({
    "format": "sdc-iden-os-tflite-v1",
    "created": datetime.now().isoformat(timespec="seconds"),
    "source_onnx": {"run": run_dir.name, "release_id": manifest["release_id"], "file": RELEASE_ONNX, "sha256": manifest["files"][RELEASE_ONNX]["sha256"]},
    "input": {"name": IO_NAMES[0], "dtype": "string", "shape": [-1, len(COLS)], "columns": COLS},
    "output": {"name": IO_NAMES[1], "dtype": "string", "shape": [-1, len(HEADS)], "columns": list(HEADS)},
    "thresholds_override": False,
    "runtime": {"requires": "LiteRT + Flex delegate (SELECT_TF_OPS)", "flex_ops": FLEX_OPS,
                "builtin_ops": [op for op in OPS if op not in FLEX_OPS]},
})
contract_bytes = json.dumps(contract, ensure_ascii=False, indent=1).encode("utf-8")
final_bytes = tflite_bytes
for key, value in ((CONTRACT_KEY, contract_bytes),
                   ("sdc.columns_sha256", contract["columns_sha256"].encode()),
                   ("sdc.labels_sha256", contract["labels_sha256"].encode()),
                   ("sdc.format", contract["format"].encode())):
    final_bytes = embed_metadata(final_bytes, key, value)

assert read_metadata(final_bytes, CONTRACT_KEY) == contract_bytes
_final = make_interpreter(final_bytes)
assert (_final.get_input_details()[0]["name"], _final.get_output_details()[0]["name"]) == IO_NAMES
if FLEX_OK:
    check = MATRICES["capture"]
    assert (run_tflite(check, final_bytes) == run_tflite(check)).all()

(release_dir / RELEASE_TFLITE).write_bytes(final_bytes)
(release_dir / "sdc_os_tflite_contract.json").write_bytes(contract_bytes)
np.savez_compressed(release_dir / "parity_inputs.npz", names=np.array(list(MATRICES)),
                    **{f"{k}/input": v.astype(str) for k, v in MATRICES.items()},
                    **{f"{k}/expected": v for k, v in EXPECTED.items()})
manifest["files"][RELEASE_TFLITE] = {
    "made_by": "13_export_release_tflite", "runtime": "LiteRT + Flex delegate", "contract_format": contract["format"],
    "sha256": hashlib.sha256(final_bytes).hexdigest(), "bytes": len(final_bytes), "created": contract["created"],
    "tensorflow": tf.__version__, "flex_ops": FLEX_OPS,
    "parity_vs_onnx": parity.reset_index().to_dict("records"),
    "parity_checked_on": "file .tflite" if FLEX_OK else "graph TF (file .tflite: parity_inputs.npz)",
}
(release_dir / RELEASE_MANIFEST).write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=float),
                                           encoding="utf-8")
for p in sorted(release_dir.iterdir()):
    print(f"{p.stat().st_size / 1024:9.1f} KiB  {p}")

      2.5 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\manifest.json
    393.8 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\parity_inputs.npz
   2284.5 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\sdc_os.onnx
   3257.9 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\sdc_os.tflite
     12.6 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\sdc_os_contract.json
     13.2 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\sdc_os_tflite_contract.json
